In [1]:
from faker import Faker
import pandas as pd

# Initialize Faker
fake = Faker()

# Define number of trades
num_of_trades = 20000

# Define allowed currencies
allowed_currencies = ['USD', 'GBP', 'JPY', 'EUR', 'CAD', 'CHF', 'AUD']

# Generate trade data
data = {
    'Trade_ID': [fake.uuid4() for _ in range(num_of_trades)],
    'Trade_Type': ['FX Swap' for _ in range(num_of_trades)], 
    'Counterparty_ID': [fake.uuid4() for _ in range(num_of_trades)],
    'Netting_Set_ID': [fake.uuid4() for _ in range(num_of_trades)],
    'Leg_1_Leg_Type': [fake.random_element(elements=('Buy', 'Sell')) for _ in range(num_of_trades)],
    'Leg_1_Currency': [fake.random_element(elements=allowed_currencies) for _ in range(num_of_trades)],
    'Leg_1_Notional': [fake.random_number(digits=7) for _ in range(num_of_trades)], 
    'Leg_2_Leg_Type': [],
    'Leg_2_Currency': [],
    'Leg_2_Notional': [fake.random_number(digits=7) for _ in range(num_of_trades)],
    'Start_Date': [fake.date_between(start_date='-1y', end_date='today') for _ in range(num_of_trades)],
    'End_Date': [fake.date_between(start_date='today', end_date='+1y') for _ in range(num_of_trades)]
}

# Ensure Leg_2_Currency is different from Leg_1_Currency
for i in range(num_of_trades):
    leg_1_currency = data['Leg_1_Currency'][i]
    leg_2_currency = fake.random_element(elements=[currency for currency in allowed_currencies if currency != leg_1_currency])
    data['Leg_2_Currency'].append(leg_2_currency)

# Set Leg_2_Leg_Type based on Leg_1_Leg_Type
for leg_1_type in data['Leg_1_Leg_Type']:
    if leg_1_type == 'Buy':
        data['Leg_2_Leg_Type'].append('Sell')
    else:
        data['Leg_2_Leg_Type'].append('Buy')

# Create F22 Dataframe 
f22_fx_swap_data = pd.DataFrame(data)

f22_fx_swap_data['Maturity'] = (pd.to_datetime(f22_fx_swap_data['End_Date']) - pd.to_datetime(f22_fx_swap_data['Start_Date'])).dt.days

pfe_data = {}
for counterparty_id in f22_fx_swap_data['Counterparty_ID']:
    pfe_data[counterparty_id] = {f'PFE_Day_{day}': fake.random_number(digits=5) for day in range(1, 31)}

f22_fx_swap_data['PFE_Values'] = f22_fx_swap_data['Counterparty_ID'].map(pfe_data)

In [2]:
f22_fx_swap_data

,Trade_ID,Trade_Type,Counterparty_ID,Netting_Set_ID,Leg_1_Leg_Type,Leg_1_Currency,Leg_1_Notional,Leg_2_Leg_Type,Leg_2_Currency,Leg_2_Notional,Start_Date,End_Date,Maturity,PFE_Values
0,3f735075-8703-4093-8658-3ddba5fce602,FX Swap,de040e0d-708d-4aec-b9e4-80b88ebb85e6,0363c5dd-36eb-4747-b685-6f7a8ec83148,Buy,AUD,7116412,Sell,CHF,7848226,2025-02-08,2025-10-26,260,"{'PFE_Day_1': 20514, 'PFE_Day_2': 3398, 'PFE_D..."
1,cfb1063c-905e-472d-b2c9-18f8991b937d,FX Swap,40846bdc-7275-47b5-9241-e847552fa766,92f9030a-5e8f-424e-afb2-3d6dac19492f,Sell,GBP,4400934,Buy,JPY,2614392,2024-11-19,2025-12-27,403,"{'PFE_Day_1': 64024, 'PFE_Day_2': 568, 'PFE_Da..."
2,9d2121e1-73f8-46b1-a141-ac187490b7f0,FX Swap,3c8fa215-836d-4b93-bbe6-4d8ff7f50940,b6ebb046-33d1-4ded-b194-bf3991c3a5c7,Sell,CHF,5064948,Buy,AUD,9127208,2024-09-08,2025-06-27,292,"{'PFE_Day_1': 29418, 'PFE_Day_2': 75418, 'PFE_..."
3,9d602f63-7034-4628-b18d-8aeb844516d7,FX Swap,0daf659b-b7cc-4425-a12b-02a890400c1b,e0fe5beb-a2b2-43d4-8862-89101ade8023,Buy,JPY,5500882,Sell,EUR,3887562,2024-10-09,2025-04-27,200,"{'PFE_Day_1': 64333, 'PFE_Day_2': 89001, 'PFE_..."
4,cbfe8d57-4935-4f23-b6a1-5eb876c7dacd,FX Swap,7884443a-d082-4f41-b059-d672f332b1a3,ed8e5817-bae1-4725-b1f5-109d5fd5701e,Sell,AUD,5661415,Buy,CHF,4243324,2024-05-20,2025-06-28,404,"{'PFE_Day_1': 57870, 'PFE_Day_2': 61105, 'PFE_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,a91862ab-bef7-47e5-bb1e-25586c53910f,FX Swap,1d86afe7-3e8a-4941-972b-a6c026a9a5ec,dcb2eb75-6601-477c-9813-d695a22da4fe,Sell,CHF,569308,Buy,GBP,8973362,2024-10-21,2025-08-12,295,"{'PFE_Day_1': 87150, 'PFE_Day_2': 79616, 'PFE_..."
19996,ac1e082a-94cc-41cb-b923-7bb823694a60,FX Swap,8ac27f69-bdc9-4a4e-9002-58fe6c6b50c9,28b593a7-13ed-444c-b4eb-b127370b7ea1,Sell,EUR,522326,Buy,JPY,7033175,2024-07-18,2025-08-23,401,"{'PFE_Day_1': 79227, 'PFE_Day_2': 23613, 'PFE_..."
19997,c600a580-5da4-4488-a470-ebdd1f543810,FX Swap,6fffafe8-50f8-4bb6-ac4a-19757740a1f6,d3a1cd22-6e0a-40d0-b371-9994669d3020,Buy,EUR,6218331,Sell,JPY,2565693,2024-08-28,2025-08-28,365,"{'PFE_Day_1': 42581, 'PFE_Day_2': 99102, 'PFE_..."
19998,c86f7255-3390-4d13-bcff-f0b2674c16c0,FX Swap,377d3dad-3f91-4039-8e1d-7449ebd603ba,1d10511e-fd0a-4065-aecb-5044cc4702d6,Buy,GBP,3785047,Sell,JPY,9776676,2024-12-22,2025-10-19,301,"{'PFE_Day_1': 75351, 'PFE_Day_2': 38315, 'PFE_..."


In [3]:
quic_fx_swap_data = f22_fx_swap_data.copy()

quic_pfe_data = {}
for counterparty_id in quic_fx_swap_data['Counterparty_ID']:
    quic_pfe_data[counterparty_id] = {f'PFE_Day_{day}': fake.random_number(digits=5) for day in range(1,31)}
    
quic_fx_swap_data['PFE_Values'] = quic_fx_swap_data['Counterparty_ID'].map(quic_pfe_data)


In [4]:
quic_fx_swap_data

,Trade_ID,Trade_Type,Counterparty_ID,Netting_Set_ID,Leg_1_Leg_Type,Leg_1_Currency,Leg_1_Notional,Leg_2_Leg_Type,Leg_2_Currency,Leg_2_Notional,Start_Date,End_Date,Maturity,PFE_Values
0,3f735075-8703-4093-8658-3ddba5fce602,FX Swap,de040e0d-708d-4aec-b9e4-80b88ebb85e6,0363c5dd-36eb-4747-b685-6f7a8ec83148,Buy,AUD,7116412,Sell,CHF,7848226,2025-02-08,2025-10-26,260,"{'PFE_Day_1': 48210, 'PFE_Day_2': 26978, 'PFE_..."
1,cfb1063c-905e-472d-b2c9-18f8991b937d,FX Swap,40846bdc-7275-47b5-9241-e847552fa766,92f9030a-5e8f-424e-afb2-3d6dac19492f,Sell,GBP,4400934,Buy,JPY,2614392,2024-11-19,2025-12-27,403,"{'PFE_Day_1': 31342, 'PFE_Day_2': 2161, 'PFE_D..."
2,9d2121e1-73f8-46b1-a141-ac187490b7f0,FX Swap,3c8fa215-836d-4b93-bbe6-4d8ff7f50940,b6ebb046-33d1-4ded-b194-bf3991c3a5c7,Sell,CHF,5064948,Buy,AUD,9127208,2024-09-08,2025-06-27,292,"{'PFE_Day_1': 24667, 'PFE_Day_2': 88284, 'PFE_..."
3,9d602f63-7034-4628-b18d-8aeb844516d7,FX Swap,0daf659b-b7cc-4425-a12b-02a890400c1b,e0fe5beb-a2b2-43d4-8862-89101ade8023,Buy,JPY,5500882,Sell,EUR,3887562,2024-10-09,2025-04-27,200,"{'PFE_Day_1': 60890, 'PFE_Day_2': 13007, 'PFE_..."
4,cbfe8d57-4935-4f23-b6a1-5eb876c7dacd,FX Swap,7884443a-d082-4f41-b059-d672f332b1a3,ed8e5817-bae1-4725-b1f5-109d5fd5701e,Sell,AUD,5661415,Buy,CHF,4243324,2024-05-20,2025-06-28,404,"{'PFE_Day_1': 42220, 'PFE_Day_2': 32460, 'PFE_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,a91862ab-bef7-47e5-bb1e-25586c53910f,FX Swap,1d86afe7-3e8a-4941-972b-a6c026a9a5ec,dcb2eb75-6601-477c-9813-d695a22da4fe,Sell,CHF,569308,Buy,GBP,8973362,2024-10-21,2025-08-12,295,"{'PFE_Day_1': 771, 'PFE_Day_2': 36574, 'PFE_Da..."
19996,ac1e082a-94cc-41cb-b923-7bb823694a60,FX Swap,8ac27f69-bdc9-4a4e-9002-58fe6c6b50c9,28b593a7-13ed-444c-b4eb-b127370b7ea1,Sell,EUR,522326,Buy,JPY,7033175,2024-07-18,2025-08-23,401,"{'PFE_Day_1': 17962, 'PFE_Day_2': 4734, 'PFE_D..."
19997,c600a580-5da4-4488-a470-ebdd1f543810,FX Swap,6fffafe8-50f8-4bb6-ac4a-19757740a1f6,d3a1cd22-6e0a-40d0-b371-9994669d3020,Buy,EUR,6218331,Sell,JPY,2565693,2024-08-28,2025-08-28,365,"{'PFE_Day_1': 23130, 'PFE_Day_2': 48513, 'PFE_..."
19998,c86f7255-3390-4d13-bcff-f0b2674c16c0,FX Swap,377d3dad-3f91-4039-8e1d-7449ebd603ba,1d10511e-fd0a-4065-aecb-5044cc4702d6,Buy,GBP,3785047,Sell,JPY,9776676,2024-12-22,2025-10-19,301,"{'PFE_Day_1': 41899, 'PFE_Day_2': 48147, 'PFE_..."


In [5]:
# Extract PFE Values into separate DataFrame
pfe_values_f22 = pd.DataFrame(f22_fx_swap_data['PFE_Values'].tolist())
pfe_values_quic = pd.DataFrame(quic_fx_swap_data['PFE_Values'].tolist())

# Calculate day-to-day differences
pfe_diff_f22 = pfe_values_f22.diff(axis=1).fillna(0)
pfe_diff_quic = pfe_values_quic.diff(axis=1).fillna(0)

# Add the differences back to the original dataframe 
f22_fx_swap_data['PFE_Day_to_Day_Diff'] = pfe_diff_f22.to_dict(orient='records')
quic_fx_swap_data['PFE_Day_to_Day_Diff'] = pfe_diff_quic.to_dict(orient='records')

f22_fx_swap_data

,Trade_ID,Trade_Type,Counterparty_ID,Netting_Set_ID,Leg_1_Leg_Type,Leg_1_Currency,Leg_1_Notional,Leg_2_Leg_Type,Leg_2_Currency,Leg_2_Notional,Start_Date,End_Date,Maturity,PFE_Values,PFE_Day_to_Day_Diff
0,3f735075-8703-4093-8658-3ddba5fce602,FX Swap,de040e0d-708d-4aec-b9e4-80b88ebb85e6,0363c5dd-36eb-4747-b685-6f7a8ec83148,Buy,AUD,7116412,Sell,CHF,7848226,2025-02-08,2025-10-26,260,"{'PFE_Day_1': 20514, 'PFE_Day_2': 3398, 'PFE_D...","{'PFE_Day_1': 0.0, 'PFE_Day_2': -17116, 'PFE_D..."
1,cfb1063c-905e-472d-b2c9-18f8991b937d,FX Swap,40846bdc-7275-47b5-9241-e847552fa766,92f9030a-5e8f-424e-afb2-3d6dac19492f,Sell,GBP,4400934,Buy,JPY,2614392,2024-11-19,2025-12-27,403,"{'PFE_Day_1': 64024, 'PFE_Day_2': 568, 'PFE_Da...","{'PFE_Day_1': 0.0, 'PFE_Day_2': -63456, 'PFE_D..."
2,9d2121e1-73f8-46b1-a141-ac187490b7f0,FX Swap,3c8fa215-836d-4b93-bbe6-4d8ff7f50940,b6ebb046-33d1-4ded-b194-bf3991c3a5c7,Sell,CHF,5064948,Buy,AUD,9127208,2024-09-08,2025-06-27,292,"{'PFE_Day_1': 29418, 'PFE_Day_2': 75418, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': 46000, 'PFE_Da..."
3,9d602f63-7034-4628-b18d-8aeb844516d7,FX Swap,0daf659b-b7cc-4425-a12b-02a890400c1b,e0fe5beb-a2b2-43d4-8862-89101ade8023,Buy,JPY,5500882,Sell,EUR,3887562,2024-10-09,2025-04-27,200,"{'PFE_Day_1': 64333, 'PFE_Day_2': 89001, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': 24668, 'PFE_Da..."
4,cbfe8d57-4935-4f23-b6a1-5eb876c7dacd,FX Swap,7884443a-d082-4f41-b059-d672f332b1a3,ed8e5817-bae1-4725-b1f5-109d5fd5701e,Sell,AUD,5661415,Buy,CHF,4243324,2024-05-20,2025-06-28,404,"{'PFE_Day_1': 57870, 'PFE_Day_2': 61105, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': 3235, 'PFE_Day..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,a91862ab-bef7-47e5-bb1e-25586c53910f,FX Swap,1d86afe7-3e8a-4941-972b-a6c026a9a5ec,dcb2eb75-6601-477c-9813-d695a22da4fe,Sell,CHF,569308,Buy,GBP,8973362,2024-10-21,2025-08-12,295,"{'PFE_Day_1': 87150, 'PFE_Day_2': 79616, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': -7534, 'PFE_Da..."
19996,ac1e082a-94cc-41cb-b923-7bb823694a60,FX Swap,8ac27f69-bdc9-4a4e-9002-58fe6c6b50c9,28b593a7-13ed-444c-b4eb-b127370b7ea1,Sell,EUR,522326,Buy,JPY,7033175,2024-07-18,2025-08-23,401,"{'PFE_Day_1': 79227, 'PFE_Day_2': 23613, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': -55614, 'PFE_D..."
19997,c600a580-5da4-4488-a470-ebdd1f543810,FX Swap,6fffafe8-50f8-4bb6-ac4a-19757740a1f6,d3a1cd22-6e0a-40d0-b371-9994669d3020,Buy,EUR,6218331,Sell,JPY,2565693,2024-08-28,2025-08-28,365,"{'PFE_Day_1': 42581, 'PFE_Day_2': 99102, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': 56521, 'PFE_Da..."
19998,c86f7255-3390-4d13-bcff-f0b2674c16c0,FX Swap,377d3dad-3f91-4039-8e1d-7449ebd603ba,1d10511e-fd0a-4065-aecb-5044cc4702d6,Buy,GBP,3785047,Sell,JPY,9776676,2024-12-22,2025-10-19,301,"{'PFE_Day_1': 75351, 'PFE_Day_2': 38315, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': -37036, 'PFE_D..."


In [6]:
quic_fx_swap_data

,Trade_ID,Trade_Type,Counterparty_ID,Netting_Set_ID,Leg_1_Leg_Type,Leg_1_Currency,Leg_1_Notional,Leg_2_Leg_Type,Leg_2_Currency,Leg_2_Notional,Start_Date,End_Date,Maturity,PFE_Values,PFE_Day_to_Day_Diff
0,3f735075-8703-4093-8658-3ddba5fce602,FX Swap,de040e0d-708d-4aec-b9e4-80b88ebb85e6,0363c5dd-36eb-4747-b685-6f7a8ec83148,Buy,AUD,7116412,Sell,CHF,7848226,2025-02-08,2025-10-26,260,"{'PFE_Day_1': 48210, 'PFE_Day_2': 26978, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': -21232, 'PFE_D..."
1,cfb1063c-905e-472d-b2c9-18f8991b937d,FX Swap,40846bdc-7275-47b5-9241-e847552fa766,92f9030a-5e8f-424e-afb2-3d6dac19492f,Sell,GBP,4400934,Buy,JPY,2614392,2024-11-19,2025-12-27,403,"{'PFE_Day_1': 31342, 'PFE_Day_2': 2161, 'PFE_D...","{'PFE_Day_1': 0.0, 'PFE_Day_2': -29181, 'PFE_D..."
2,9d2121e1-73f8-46b1-a141-ac187490b7f0,FX Swap,3c8fa215-836d-4b93-bbe6-4d8ff7f50940,b6ebb046-33d1-4ded-b194-bf3991c3a5c7,Sell,CHF,5064948,Buy,AUD,9127208,2024-09-08,2025-06-27,292,"{'PFE_Day_1': 24667, 'PFE_Day_2': 88284, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': 63617, 'PFE_Da..."
3,9d602f63-7034-4628-b18d-8aeb844516d7,FX Swap,0daf659b-b7cc-4425-a12b-02a890400c1b,e0fe5beb-a2b2-43d4-8862-89101ade8023,Buy,JPY,5500882,Sell,EUR,3887562,2024-10-09,2025-04-27,200,"{'PFE_Day_1': 60890, 'PFE_Day_2': 13007, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': -47883, 'PFE_D..."
4,cbfe8d57-4935-4f23-b6a1-5eb876c7dacd,FX Swap,7884443a-d082-4f41-b059-d672f332b1a3,ed8e5817-bae1-4725-b1f5-109d5fd5701e,Sell,AUD,5661415,Buy,CHF,4243324,2024-05-20,2025-06-28,404,"{'PFE_Day_1': 42220, 'PFE_Day_2': 32460, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': -9760, 'PFE_Da..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,a91862ab-bef7-47e5-bb1e-25586c53910f,FX Swap,1d86afe7-3e8a-4941-972b-a6c026a9a5ec,dcb2eb75-6601-477c-9813-d695a22da4fe,Sell,CHF,569308,Buy,GBP,8973362,2024-10-21,2025-08-12,295,"{'PFE_Day_1': 771, 'PFE_Day_2': 36574, 'PFE_Da...","{'PFE_Day_1': 0.0, 'PFE_Day_2': 35803, 'PFE_Da..."
19996,ac1e082a-94cc-41cb-b923-7bb823694a60,FX Swap,8ac27f69-bdc9-4a4e-9002-58fe6c6b50c9,28b593a7-13ed-444c-b4eb-b127370b7ea1,Sell,EUR,522326,Buy,JPY,7033175,2024-07-18,2025-08-23,401,"{'PFE_Day_1': 17962, 'PFE_Day_2': 4734, 'PFE_D...","{'PFE_Day_1': 0.0, 'PFE_Day_2': -13228, 'PFE_D..."
19997,c600a580-5da4-4488-a470-ebdd1f543810,FX Swap,6fffafe8-50f8-4bb6-ac4a-19757740a1f6,d3a1cd22-6e0a-40d0-b371-9994669d3020,Buy,EUR,6218331,Sell,JPY,2565693,2024-08-28,2025-08-28,365,"{'PFE_Day_1': 23130, 'PFE_Day_2': 48513, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': 25383, 'PFE_Da..."
19998,c86f7255-3390-4d13-bcff-f0b2674c16c0,FX Swap,377d3dad-3f91-4039-8e1d-7449ebd603ba,1d10511e-fd0a-4065-aecb-5044cc4702d6,Buy,GBP,3785047,Sell,JPY,9776676,2024-12-22,2025-10-19,301,"{'PFE_Day_1': 41899, 'PFE_Day_2': 48147, 'PFE_...","{'PFE_Day_1': 0.0, 'PFE_Day_2': 6248, 'PFE_Day..."


In [7]:
import pandas as pd
from faker import Faker

fake = Faker()
allowed_currencies = ['USD', 'GBP', 'JPY', 'EUR', 'CAD', 'CHF', 'AUD']

# Create a DataFrame to hold spot rates for all currency pairs
spot_rates_matrix = pd.DataFrame(index=allowed_currencies, columns=allowed_currencies, dtype=float)

# Populate with random values
for base in allowed_currencies:
    for quote in allowed_currencies:
        if base == quote:
            spot_rates_matrix.loc[base, quote] = 1.0
        else:
            rand_value = round(fake.random_number(digits=2) + fake.random_number(digits=5) / 100000, 5)
            spot_rates_matrix.loc[base, quote] = rand_value

spot_rates_matrix


,USD,GBP,JPY,EUR,CAD,CHF,AUD
USD,1.00000,38.39599,67.64218,5.20854,13.39322,35.93156,74.92144
GBP,3.01924,1.00000,82.44615,15.99184,61.36236,14.27099,80.24265
JPY,89.41957,73.68891,1.00000,71.62518,78.07722,51.17047,29.55736
EUR,87.22894,91.36561,43.80557,1.00000,55.20256,90.76023,99.15834
CAD,53.50151,17.59032,22.05613,6.06754,1.00000,43.86581,33.16867
CHF,5.81913,66.12959,82.84618,61.63141,32.21265,1.00000,17.75450
AUD,11.93584,98.50632,20.21353,78.68111,67.28541,13.89059,1.00000


In [8]:
# Create a volatility correlation matrix 
import numpy as np
import pandas as pd

allowed_currencies = ['USD','GBP','JPY','EUR','CAD','CHF','AUD']

# Create an initial random matrix
size = len(allowed_currencies)
random_matrix = np.random.rand(size, size)

# Make it symmetric
correlation_matrix = (random_matrix + random_matrix.T) / 2

# Set diagonal to 1
np.fill_diagonal(correlation_matrix, 1)

# Create DataFrame
volatility_correlation_df = pd.DataFrame(
    correlation_matrix, 
    index=allowed_currencies, 
    columns=allowed_currencies
)

volatility_correlation_df


,USD,GBP,JPY,EUR,CAD,CHF,AUD
USD,1.000000,0.607187,0.291696,0.485135,0.728805,0.322456,0.130251
GBP,0.607187,1.000000,0.494928,0.179117,0.203986,0.515670,0.133322
JPY,0.291696,0.494928,1.000000,0.402324,0.807622,0.618047,0.356513
EUR,0.485135,0.179117,0.402324,1.000000,0.263184,0.176207,0.501682
CAD,0.728805,0.203986,0.807622,0.263184,1.000000,0.427608,0.671844
CHF,0.322456,0.515670,0.618047,0.176207,0.427608,1.000000,0.468746
AUD,0.130251,0.133322,0.356513,0.501682,0.671844,0.468746,1.000000


In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


# For merging, we can assume the rows are aligned by Counterparty_ID (or use an explicit merge)
merged_df = f22_fx_swap_data.copy()
merged_df['QUIC_PFE_Day_to_Day_Diff'] = quic_fx_swap_data['PFE_Day_to_Day_Diff']

# Define a function to compute the target value:
# Here, we compute the 95th percentile of the difference between the F22 and QUIC day-to-day PFE differences.
def compute_target(row):
    f22_values = np.array(list(row['PFE_Day_to_Day_Diff'].values()))
    quic_values = np.array(list(row['QUIC_PFE_Day_to_Day_Diff'].values()))
    # Calculate the difference between the two engines for each day
    diff_values = f22_values - quic_values
    # Return the 95th percentile of these differences
    return np.percentile(diff_values, 95)

merged_df['PFE_Diff_Target'] = merged_df.apply(compute_target, axis=1)

# Define functions to lookup spot rates and volatility correlation for each trade
def get_spot_rate(row):
    return spot_rates_matrix.loc[row['Leg_1_Currency'], row['Leg_2_Currency']]

def get_vol_corr(row):
    return volatility_correlation_df.loc[row['Leg_1_Currency'], row['Leg_2_Currency']]


# Add new columns to your merged dataframe
merged_df['Spot_Rate'] = merged_df.apply(get_spot_rate, axis=1)
merged_df['Vol_Correlation'] = merged_df.apply(get_vol_corr, axis=1)

# Now, when selecting features, include these additional numeric features.
features = merged_df[['Maturity', 'Spot_Rate', 'Vol_Correlation', 'Leg_1_Currency', 'Leg_2_Currency']]

# One-hot encode the categorical currency features
features_encoded = pd.get_dummies(features, columns=['Leg_1_Currency', 'Leg_2_Currency'])

# Your target variable remains the same (the 95th percentile of the PFE difference)
target = merged_df['PFE_Diff_Target']

# Split the data into training and testing sets.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(features_encoded, target, test_size=0.2, random_state=42)

# Build the random forest model.
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions and evaluate the model.
from sklearn.metrics import mean_squared_error
y_pred = rf_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)

# Display feature importance
feature_importances = pd.Series(rf_model.feature_importances_, index=features_encoded.columns)
print("Feature Importances:\n", feature_importances.sort_values(ascending=False))


Mean Squared Error: 340841844.9952147
Feature Importances:
 Maturity              0.693382
Spot_Rate             0.074095
Vol_Correlation       0.069280
Leg_2_Currency_CHF    0.013943
Leg_1_Currency_GBP    0.013806
Leg_1_Currency_AUD    0.012484
Leg_2_Currency_CAD    0.012462
Leg_2_Currency_GBP    0.012266
Leg_2_Currency_JPY    0.011802
Leg_2_Currency_AUD    0.011444
Leg_2_Currency_USD    0.011344
Leg_2_Currency_EUR    0.010985
Leg_1_Currency_JPY    0.010930
Leg_1_Currency_EUR    0.010833
Leg_1_Currency_CHF    0.010591
Leg_1_Currency_USD    0.010216
Leg_1_Currency_CAD    0.010138
dtype: float64


In [10]:
# Sort trades by the PFE_Diff_Target and select the top 10 trades.
impactful_trades = merged_df.sort_values('PFE_Diff_Target', ascending=False).head(10)

# Compute the average PFE value for these top trades.
average_pfe = impactful_trades['PFE_Diff_Target'].mean()

# Select key columns for clarity: including trade ID, counterparty, PFE value, maturity, currency pairs, spot rate, and volatility correlation.
impactful_trades_summary = impactful_trades[['Trade_ID', 'Counterparty_ID', 'PFE_Diff_Target', 
                                               'Maturity', 'Leg_1_Currency', 'Leg_2_Currency', 
                                               'Spot_Rate', 'Vol_Correlation']]

# Convert the summary to a string for inclusion in the prompt.
summary_text = impactful_trades_summary.to_string(index=False)

# Create a concise summary including the average PFE value.
prompt_summary = f"""Top 10 Trades Summary:
{summary_text}

Average PFE for Top 10 Trades: {average_pfe:.2f}
"""

print(prompt_summary)


Top 10 Trades Summary:
                            Trade_ID                      Counterparty_ID  PFE_Diff_Target  Maturity Leg_1_Currency Leg_2_Currency  Spot_Rate  Vol_Correlation
4b7f3de9-8cfc-4935-86b3-3233834259c4 7b4fb060-27e1-4157-a5e7-2023dffb5dad        154509.80       340            AUD            USD   11.93584         0.130251
5623852d-43b3-4eb4-94f7-07d07bd88a60 7aeb4988-f464-4f00-b26f-22523a5f5496        153316.35       600            GBP            CHF   14.27099         0.515670
0f50b15d-6514-4313-a179-4dad6602f9c9 194f0ef0-efd4-4c54-bf6c-0869327c4940        151533.60       655            AUD            GBP   98.50632         0.133322
60d38a92-83b1-43ce-a4e4-a1c643ec2857 86c42e3d-09ab-424c-ae3b-55c4f56c96ee        147547.05       423            JPY            AUD   29.55736         0.356513
d4016f4c-b2c5-4bab-95f4-2507bd32f965 7292a035-be66-4bc1-a98f-2e7a4677412a        145820.75       416            CAD            JPY   22.05613         0.807622
028135a7-fb4e-4021-b0ea

In [11]:
prompt = f"""
I have been analyzing FX swap trades using two risk engines, F22 and QUIC. I computed the 95th percentile of the day-to-day differences in Potential Future Exposure (PFE) between these engines. In addition, I have market data in the form of a spot rate matrix and a volatility correlation matrix for various currency pairs. Lastly, I built a random forest model to predict the PFE differences.

Here are the details:

--- Top Ten Trades ---
{summary_text}


--- Spot Rate Matrix ---
{spot_rates_matrix.to_string()}

--- Volatility Correlation Matrix ---
{volatility_correlation_df.to_string()}

--- Random Forest Output ---
Mean Squared Error: {mse}
Feature Importances:
{feature_importances.sort_values(ascending=False).to_string()}

Based on this data, please provide an analysis of:
1. Which factors are driving the differences in PFE between the two risk engines?
2. What do the top ten trades suggest?
3. How the spot rates and volatility correlations might be influencing these differences.
4. Suggestions for further improvements to the model and additional features or techniques that could improve the prediction accuracy.

Please explain your reasoning in detail.
"""




In [12]:
from openai import OpenAI

client = OpenAI(api_key='api_key')

completion = client.chat.completions.create(
  model="o1-mini",
    messages=[
          {"role": "assistant", "content": "You are an expert in risk analytics and machine learning."},
        {"role": "user", "content": prompt}
    ]
)

print(completion.choices[0].message.content)


Based on the provided data and analysis, here's a comprehensive evaluation addressing your four key questions:

---

### **1. Factors Driving the Differences in PFE Between the Two Risk Engines**

**Primary Drivers Identified by the Random Forest Model:**

- **Maturity (Importance: 0.693):** 
  - **Impact:** The maturity of a trade significantly influences the Potential Future Exposure (PFE) difference between F22 and QUIC. Longer maturities generally introduce more uncertainty, leading to larger discrepancies in risk assessments between different engines due to varying modeling assumptions and time horizons.
  - **Reasoning:** Risk engines often handle time-dependent factors differently. For instance, one engine might use a more conservative approach for discounting cash flows over extended periods, while another might incorporate stochastic elements that react differently to market volatility over time.

- **Spot_Rate (Importance: 0.074):**
  - **Impact:** The current exchange rate b

In [13]:
print(completion.choices[0].message)

ChatCompletionMessage(content="Based on the provided data and analysis, here's a comprehensive evaluation addressing your four key questions:\n\n---\n\n### **1. Factors Driving the Differences in PFE Between the Two Risk Engines**\n\n**Primary Drivers Identified by the Random Forest Model:**\n\n- **Maturity (Importance: 0.693):** \n  - **Impact:** The maturity of a trade significantly influences the Potential Future Exposure (PFE) difference between F22 and QUIC. Longer maturities generally introduce more uncertainty, leading to larger discrepancies in risk assessments between different engines due to varying modeling assumptions and time horizons.\n  - **Reasoning:** Risk engines often handle time-dependent factors differently. For instance, one engine might use a more conservative approach for discounting cash flows over extended periods, while another might incorporate stochastic elements that react differently to market volatility over time.\n\n- **Spot_Rate (Importance: 0.074):**\

# Further Fine Tuning Needed